In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sys
from pathlib import Path
from typing import Any

In [ ]:
project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
print(f"Project root: {project_root}")
UNSW_PATH = project_root / "data" / "UNSW-NB15" / "UNSW_NB15_training-set.csv"

In [ ]:
FEATURES = ["dur", "sbytes", "dbytes", "sload", "dload", "rate"]
LABEL = "label"

In [ ]:
df = pd.read_csv(UNSW_PATH)
X = df[FEATURES].values
y = df[LABEL].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
class IDS(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = IDS()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

Xt = torch.tensor(X_train, dtype=torch.float32)
yt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

for _ in range(10):
    optimizer.zero_grad()
    loss = criterion(model(Xt), yt)
    loss.backward()
    optimizer.step()


In [ ]:
def fgsm_attack(model, x, y, eps=0.15):
    x_adv = x.clone().detach().requires_grad_(True)
    loss = criterion(model(x_adv), y)
    loss.backward()
    return (x_adv + eps * x_adv.grad.sign()).detach()


In [ ]:

def gaussian_noise_sample(x, scale=1.0):

    noise = torch.randn_like(x) * scale
    return noise


In [ ]:
def radar_plot(orig, adv, features, title):
    angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])

    orig = np.concatenate([orig, orig[:1]])
    adv = np.concatenate([adv, adv[:1]])

    fig = plt.figure(figsize=(5, 5))
    ax = plt.subplot(111, polar=True)

    ax.plot(angles, orig, label="Original", linewidth=2)
    ax.fill(angles, orig, alpha=0.3)

    #ax.plot(angles, adv, label="Adversarial", linestyle="--", linewidth=2)
    #ax.fill(angles, adv, alpha=0.2)

    ax.set_thetagrids(angles[:-1] * 180 / np.pi, features, fontsize=14)
    ax.set_title(title)
    #ax.legend(loc="upper right")
    plt.show()


In [ ]:
def plot_adv_examples(orig, adv, features, title):
    angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])

    orig = np.concatenate([orig, orig[:1]])
    adv = np.concatenate([adv, adv[:1]])

    fig = plt.figure(figsize=(5, 5))
    ax = plt.subplot(111, polar=True)

    #ax.plot(angles, orig, label="Original", linewidth=2)
    #ax.fill(angles, orig, alpha=0.3)

    ax.plot(angles, adv, label="Adversarial", linestyle="--", linewidth=2)
    ax.fill(angles, adv, alpha=0.2)

    ax.set_thetagrids(angles[:-1] * 180 / np.pi, features, fontsize=14)
    ax.set_title(title)
    #ax.legend(loc="upper right")
    plt.show()

In [ ]:
indices = np.random.choice(len(X_test), 5, replace=False)

for i, idx in enumerate(indices):
    x = torch.tensor(X_test[idx], dtype=torch.float32).unsqueeze(0)
    y_true = torch.tensor([[y_test[idx]]], dtype=torch.float32)

    x_adv = fgsm_attack(model, x, y_true)
    x_noise = gaussian_noise_sample(x, scale=1.0)

    radar_plot(
        x.squeeze().numpy(),
        x_adv.squeeze().numpy(),
        FEATURES,
        title=f"Network Flow {i+1}"
    )
    plot_adv_examples(
        x.squeeze().numpy(),
        x_adv.squeeze().numpy(),
        FEATURES,
        title=f"Adversarial Network Flow {i+1}"
    )
    plot_adv_examples(
        x.squeeze().numpy(),
        x_noise.squeeze().numpy(),
        FEATURES,
        title=f"Gaussian Noisy Network Flow {i+1}"
    )